In [1]:
# ========== 0) 드라이브 마운트 ==========
from google.colab import drive
drive.mount('/content/drive')

# ========== 1) 경로/임포트 ==========
from pathlib import Path
import os, re, zipfile, pandas as pd

ROOT_DIR    = Path('/content/drive/MyDrive/IMU_DATA')   # 드라이브 내 작업 폴더
EXTRACT_DIR = Path('/content/IMU_DATA_extracted')       # Colab 런타임 로컬
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# ========== 2) ZIP 자동 탐색 & 해제 (없으면 스킵) ==========
zip_candidates = sorted(ROOT_DIR.glob('*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
if zip_candidates:
    ZIP_PATH = zip_candidates[0]
    print(f"[INFO] 사용할 ZIP: {ZIP_PATH.name}")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"[OK] 압축 해제 완료 → {EXTRACT_DIR.resolve()}")
else:
    print(f"[WARN] ZIP을 찾지 못했습니다: {ROOT_DIR}  (이미 해제된 폴더를 사용합니다)")

# ========== 3) label_* 폴더의 '공통 상위 경로' 자동 탐색 ==========
def _is_label_dir(p: Path):
    name = p.name.lower()
    # label_*, class_*, good/bad 형태 지원
    return (
        name.startswith('label_') or
        name.startswith('class_') or
        name in {'good','bad','positive','negative'}
    )

def _list_label_dirs(base: Path):
    # rglob로 모든 하위 폴더 탐색
    return [d for d in base.rglob('*') if d.is_dir() and _is_label_dir(d)]

def _common_parent(paths):
    # 여러 경로의 공통 상위 경로 반환
    if not paths: return None
    common = os.path.commonpath([str(p) for p in paths])
    return Path(common)

def find_dataset_root(extracted: Path, fallback_root: Path):
    # 1) 해제 경로에서 label 디렉토리 찾기
    label_dirs = _list_label_dirs(extracted)
    if label_dirs:
        parent = _common_parent([d.parent for d in label_dirs])
        print(f"[INFO] DATASET_ROOT(auto): {parent}")
        return parent
    # 2) 드라이브 루트 하위에서 직접 찾기(이미 해제된 경우)
    label_dirs = _list_label_dirs(fallback_root)
    if label_dirs:
        parent = _common_parent([d.parent for d in label_dirs])
        print(f"[INFO] DATASET_ROOT(auto from DRIVE): {parent}")
        return parent
    raise FileNotFoundError("[ERROR] 'label_*' 또는 유사 폴더를 찾지 못했습니다.")

DATASET_ROOT = find_dataset_root(EXTRACT_DIR, ROOT_DIR)

# ========== 4) 라벨 매핑 자동화 + 안전 CSV 로딩 ==========
def _infer_label_map(root: Path):
    """root 하위의 label 디렉토리를 자동 매핑: label_0→0, label_1→1, ... / 그 외는 사전식 정렬 순서대로 0..K-1"""
    subdirs = [d for d in sorted(root.iterdir()) if d.is_dir() and _is_label_dir(d)]
    if not subdirs:
        # 한 단계 더 내려가서 재탐색
        subdirs = sorted([d for d in root.rglob('*') if d.is_dir() and _is_label_dir(d)], key=lambda p: p.as_posix())
    if not subdirs:
        raise FileNotFoundError("[ERROR] 라벨 디렉토리를 찾지 못했습니다.")
    mapping = {}
    for d in subdirs:
        name = d.name
        m = re.match(r'^(?:label|class)_(\d+)$', name, flags=re.IGNORECASE)
        if m:
            y = int(m.group(1))
        else:
            # good/bad 등은 사전식 정렬 순으로 0..K-1
            # 단, 관례상 'bad/negative'를 0, 'good/positive'를 1로 맞추려면 아래 우선순위 가중치 제공
            order_bias = {'bad':0, 'negative':0, 'good':1, 'positive':1}
            y = order_bias.get(name.lower(), None)
            if y is None:
                # 임시 None이면 나중에 정렬해서 인덱스 부여
                y = None
        mapping[name] = y
    # None이 남아있으면 사전식 정렬 순서대로 빈 ID를 채움
    used = {v for v in mapping.values() if v is not None}
    next_ids = [i for i in range(len(mapping)) if i not in used]
    for k in sorted([k for k,v in mapping.items() if v is None]):
        mapping[k] = next_ids.pop(0)
    return mapping

def _read_csv_safely(path: Path, encoding_pref=('utf-8-sig','cp949','utf-8')):
    last_err = None
    for enc in encoding_pref:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_err = e
    # 마지막 시도로 인코딩 미지정
    try:
        return pd.read_csv(path)
    except Exception:
        raise last_err

def load_labeled_imu(root_dir: Path,
                     label_map='auto',
                     pattern=('*.csv', '*.CSV'),
                     verbose=True,
                     preview_files=5):
    root = Path(root_dir)
    if not root.exists():
        raise FileNotFoundError(f"[ERROR] 루트가 없습니다: {root}")

    if label_map == 'auto':
        label_map_used = _infer_label_map(root)
    else:
        label_map_used = dict(label_map)

    print(f"[INFO] 라벨 매핑: {label_map_used}")

    total_files, dfs = 0, []
    for subdir_name, y in label_map_used.items():
        d = root / subdir_name
        if not (d.exists() and d.is_dir()):
            print(f"[MISS] 폴더 없음: {d}")
            continue
        # 여러 패턴 지원
        files = []
        for pat in pattern:
            files += list(d.glob(pat))
        files = sorted(set(files))
        n = len(files); total_files += n
        print(f"[OK] {d.name} → {n}개 파일")
        if verbose and n>0:
            for f in files[:preview_files]:
                print(f"    • {f.name}")
            if n > preview_files:
                print(f"    • ... (총 {n}개)")
        for f in files:
            try:
                df = _read_csv_safely(f)
            except Exception as e:
                print(f"[WARN] 읽기 실패: {f.name} ({e})"); continue
            # 메타정보
            df['label']       = y
            df['source_file'] = f.name
            df['source_dir']  = subdir_name
            dfs.append(df)

    if total_files == 0 or not dfs:
        raise FileNotFoundError("[ERROR] CSV를 찾지 못했습니다. 경로/패턴을 확인하세요.")

    data = pd.concat(dfs, axis=0, ignore_index=True, sort=False)

    # 로딩 품질 점검 로그(선택)
    def _normalize_name(s):
        s = re.sub(r'[^A-Za-z0-9]+',' ', str(s)).strip().lower()
        return re.sub(r'\s+',' ', s)
    cols_norm = {_normalize_name(c): c for c in data.columns}
    move_candidates = [c for c in data.columns if _normalize_name(c) in ['ak','move','rep','segment','cycle','trial','action']]
    acc_candidates  = [c for c in data.columns if any(k in _normalize_name(c) for k in ['acc','accelerometer'])]
    gyr_candidates  = [c for c in data.columns if any(k in _normalize_name(c) for k in ['gyr','gyro'])]

    print("\n[SUMMARY]")
    print(f" - 총 CSV 파일 수: {total_files}개")
    print(f" - 총 로우 수: {len(data):,}")
    print(f" - 라벨 분포:\n{data['label'].value_counts(dropna=False).to_string()}")
    if move_candidates:
        print(f" - move 관련 열 후보: {move_candidates}")
    else:
        print(" - [WARN] move(AK) 열을 찾지 못했습니다. 전처리 전에 열 이름을 확인하세요.")
    print(f" - 가속도 열 후보: {acc_candidates[:6]}")
    print(f" - 자이로 열 후보: {gyr_candidates[:6]}")

    return data, label_map_used

# ========== 5) 실제 로딩 ==========
imu_df, label_map_used = load_labeled_imu(
    DATASET_ROOT,
    label_map='auto',            # 필요하면 {'label_0':0,'label_1':1}로 고정 가능
    pattern=('*.csv','*.CSV'),
    verbose=True,
    preview_files=5
)

# ========== 6) (옵션) 업로드 CSV 폴백 병합 ==========
# Colab 외부에서 개별 CSV를 올려둔 경우(예: ChatGPT에서 제공) 병합
fallback_csvs = [Path('/mnt/data/IMU_Label_Plus_ALL.csv'), Path('/mnt/data/IMU_label_1_jungro.csv')]
fallback_exist = [p for p in fallback_csvs if p.exists()]
if fallback_exist:
    add_dfs = []
    for p in fallback_exist:_


Mounted at /content/drive
[INFO] 사용할 ZIP: IMU_all.zip
[OK] 압축 해제 완료 → /content/IMU_DATA_extracted
[INFO] DATASET_ROOT(auto): /content/IMU_DATA_extracted
[INFO] 라벨 매핑: {'label_0': 0, 'label_1': 1}
[OK] label_0 → 1개 파일
    • IMU_Label_Plus_ALL.csv
[OK] label_1 → 1개 파일
    • IMU_label_1_jungro.csv

[SUMMARY]
 - 총 CSV 파일 수: 2개
 - 총 로우 수: 45,143
 - 라벨 분포:
label
0    36535
1     8608
 - move 관련 열 후보: ['move']
 - 가속도 열 후보: []
 - 자이로 열 후보: []


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] 발견 파일 수: 12
  • _merged_all_with_labels.csv
  • _merged_all_with_labels_meta.csv
  • length_histogram.csv
  • label_counts.csv
  • group_move_counts.csv
[SKIP] _merged_all_with_labels.csv → missing ['ax', 'ay', 'az', 'gx', 'gy', 'gz'] (cols: ['Timestep', 'IMU1_tick', 'IMU1_ax', 'IMU1_ay', 'IMU1_az', 'IMU1_gx', 'IMU1_gy', 'IMU1_gz', 'IMU2_tick', 'IMU2_ax', 'IMU2_ay', 'IMU2_az'] ...)
[SKIP] _merged_all_with_labels_meta.csv → missing ['ax', 'ay', 'az', 'gx', 'gy', 'gz'] (cols: ['group', 'move_idx_in_group', 'start_idx_raw', 'end_idx_raw', 'start_idx_used', 'end_idx_used', 'length_used', 'person_id', 'label'] ...)
[SKIP] length_histogram.csv → missing ['ax', 'ay', 'az', 'gx', 'gy', 'gz'] (cols: ['length_used', 'count'] ...)
[SKIP] label_counts.csv → missing ['ax', 'ay', 'az', 'gx', 'gy', 'gz'] (cols: ['Unnamed: 0', 'count'] ...)
[SKIP] group_move_counts.cs

In [9]:
!pip install openpyxl

In [2]:
# ========== 7) EKF 열 추가 배치 파이프라인 (MOVE 단위 리셋) ==========
import numpy as np, pandas as pd, math, re, os, zipfile
from pathlib import Path

# 앞에서 정의된 변수들 사용: ROOT_DIR, EXTRACT_DIR, DATASET_ROOT
OUT_DIR = (EXTRACT_DIR / "ekf_appended"); OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------- 설정 --------
FS = 50.0
PREFER_IMU = "IMU1"
ACC_UNIT = "auto"              # 'auto'|'ms2'|'g'
GYRO_IS_DEG = "auto"           # 'auto'|True|False
G = 9.80665
REINIT_PER_MOVE = True         # ✅ MOVE마다 상태 리셋 (원하는 동작)
INIT_STATIC_SEC = 0.3          # MOVE 시작부 평균으로 q0/bias 추정에 쓸 길이(초)

ALIASES_T = ['t','time','timestamp','Timestep','IMU1_tick','tick','Time','TIME']
IMU_KEYS  = ['ax','ay','az','gx','gy','gz']
MOVE_COL_CANDS = ['move','Move','MOVE','ak','AK','rep','segment','trial','cycle']

def read_csv_safely(p: Path, encs=('utf-8-sig','cp949','utf-8')):
    last = None
    for enc in encs:
        try:
            return pd.read_csv(p, encoding=enc)
        except Exception as e:
            last = e
    try:
        return pd.read_csv(p, header=None)
    except Exception:
        raise last

def pick_move_col(df: pd.DataFrame):
    for c in MOVE_COL_CANDS:
        if c in df.columns: return c
    # 느슨한 매칭(열 이름 정규화)
    def norm(s): return re.sub(r'\s+',' ', re.sub('[^0-9A-Za-z]+',' ', str(s))).strip().lower()
    normmap = {norm(c): c for c in df.columns}
    for key in ['move','ak','rep','segment','trial','cycle']:
        if key in normmap: return normmap[key]
    return None  # 없으면 파일 전체를 한 덩어리로 처리

def _is_numeric_like_colnames(cols, k=8):
    k = min(len(cols), k); pat = re.compile(r'^-?\d+(\.\d+)?$')
    return all(pat.match(str(c)) for c in cols[:k])

def pick_imu_block(df: pd.DataFrame) -> pd.DataFrame:
    cols = df.columns
    def _has(prefix): return all((prefix+k) in cols for k in IMU_KEYS)
    prefer = "IMU1_" if PREFER_IMU.upper()=="IMU1" else "IMU2_"
    alt    = "IMU2_" if prefer=="IMU1_" else "IMU1_"

    if _has(prefer):
        data = {k: pd.to_numeric(df[prefer+k], errors='coerce').to_numpy() for k in IMU_KEYS}
    elif _has(alt):
        data = {k: pd.to_numeric(df[alt+k], errors='coerce').to_numpy() for k in IMU_KEYS}
    elif all(k in cols for k in IMU_KEYS):
        data = {k: pd.to_numeric(df[k], errors='coerce').to_numpy() for k in IMU_KEYS}
    else:
        # 헤더 없이 앞 6열 가정
        if df.shape[1] >= 6 and all(isinstance(c,int) for c in cols):
            use = df.iloc[:, :6].copy(); use.columns = IMU_KEYS
            data = {k: pd.to_numeric(use[k], errors='coerce').to_numpy() for k in IMU_KEYS}
        else:
            raise ValueError("ax..gz 6채널을 찾지 못했습니다.")

    # 시간열
    t = None
    for c in ALIASES_T:
        if c in cols:
            t = pd.to_numeric(df[c], errors='coerce').to_numpy(); break
    if t is None:
        t = np.arange(len(next(iter(data.values())))) / FS
    else:
        finite = np.isfinite(t)
        if finite.any():
            diffs = np.diff(t[finite]) if finite.sum()>1 else np.array([])
            if diffs.size:
                med = float(np.median(np.abs(diffs)))
                if med > 1.0/FS*10:
                    if 10 <= med <= 100: t = t / 1000.0
                    else:                 t = np.arange(len(t)) / FS
        else:
            t = np.arange(len(t)) / FS

    return pd.DataFrame({'t': t, **data})

def ensure_units(df_xyzg: pd.DataFrame) -> pd.DataFrame:
    if ACC_UNIT == 'g':
        for c in ['ax','ay','az']: df_xyzg[c] = df_xyzg[c] * G
    elif ACC_UNIT == 'auto':
        amag = np.median(np.sqrt(df_xyzg.ax**2 + df_xyzg.ay**2 + df_xyzg.az**2))
        if 0.5 <= amag <= 2.0:
            for c in ['ax','ay','az']: df_xyzg[c] = df_xyzg[c] * G
    if GYRO_IS_DEG is True:
        for c in ['gx','gy','gz']: df_xyzg[c] = np.deg2rad(df_xyzg[c])
    elif GYRO_IS_DEG == 'auto':
        g95 = np.percentile(np.abs(np.r_[df_xyzg.gx.values, df_xyzg.gy.values, df_xyzg.gz.values]), 95)
        if 10 < g95 < 500:
            for c in ['gx','gy','gz']: df_xyzg[c] = np.deg2rad(df_xyzg[c])
    return df_xyzg

# ---------- EKF ----------
def q_norm(q):
    n = np.linalg.norm(q);
    return q if n==0 else q/n

def omega(gyro):
    gx,gy,gz = gyro
    return np.array([[0, -gx, -gy, -gz],
                     [gx,  0,  gz, -gy],
                     [gy, -gz,  0,  gx],
                     [gz,  gy, -gx,  0]], dtype=float)

def rotmat_from_q(q):
    w,x,y,z = q
    return np.array([
        [1-2*(y*y+z*z), 2*(x*y - z*w), 2*(x*z + y*w)],
        [2*(x*y + z*w), 1-2*(x*x+z*z), 2*(y*z - x*w)],
        [2*(x*z - y*w), 2*(y*z + x*w), 1-2*(x*x+y*y)]
    ])

def euler_from_q(q):
    w,x,y,z = q
    yaw   = math.atan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))
    pitch = math.asin(max(-1.0, min(1.0, 2*(w*y - z*x))))
    roll  = math.atan2(2*(w*x + y*z), 1 - 2*(x*x + y*y))
    return roll, pitch, yaw

def estimate_init_q_from_acc(a):
    a = a / (np.linalg.norm(a) + 1e-12)
    pitch = np.arctan2(-a[0], np.sqrt(a[1]**2 + a[2]**2))
    roll  = np.arctan2( a[1], a[2])
    cy, sy = 1.0, 0.0        # yaw=0
    cr, sr = np.cos(roll/2), np.sin(roll/2)
    cp, sp = np.cos(pitch/2), np.sin(pitch/2)
    w = cy*cp*cr + sy*sp*sr
    x = cy*cp*sr - sy*sp*cr
    y = cy*sp*cr + sy*cp*sr
    z = sy*cp*cr - cy*sp*sr
    return q_norm(np.array([w,x,y,z]))

def ahrs_ekf_stable(acc, gyro, t, q0=None, b0=None):
    N = len(acc)
    dt_arr = np.diff(t, prepend=t[0])
    dt_arr = np.clip(dt_arr, 1e-4, 1.0)

    x = np.zeros(7); x[:4] = q_norm(q0) if q0 is not None else np.array([1.,0.,0.,0.])
    if b0 is not None: x[4:] = b0
    P = np.diag([1e-4]*4 + [1e-4]*3)

    q_proc = 1e-6
    b_rw   = 5e-3
    Q = np.diag([q_proc]*4 + [b_rw**2]*3)

    r_acc  = 1.0
    Rm     = np.diag([r_acc**2]*3)
    g_ref  = np.array([0,0,G])
    I7     = np.eye(7); epsS = 1e-9

    out = {
        'qw':np.zeros(N), 'qx':np.zeros(N), 'qy':np.zeros(N), 'qz':np.zeros(N),
        'bgx':np.zeros(N), 'bgy':np.zeros(N), 'bgz':np.zeros(N),
        'roll':np.zeros(N), 'pitch':np.zeros(N), 'yaw':np.zeros(N),
        'acc_lin_x':np.zeros(N), 'acc_lin_y':np.zeros(N), 'acc_lin_z':np.zeros(N),
    }

    for k in range(N):
        dt = dt_arr[k]
        q = x[:4]; b = x[4:]; w = gyro[k] - b

        # 예측
        Fq = np.eye(4) + 0.5*omega(w)*dt
        q_pred = q_norm(Fq @ q)
        b_pred = b
        x_pred = np.hstack([q_pred, b_pred])

        F = np.eye(7); F[:4,:4] = Fq
        P_pred = F @ P @ F.T + Q

        # 갱신
        acc_k = acc[k]; acc_norm = np.linalg.norm(acc_k)
        do_update = (acc_norm > 1e-6) and (0.6*G <= acc_norm <= 1.4*G)
        if do_update:
            z = acc_k / acc_norm
            Rb = rotmat_from_q(q_pred).T
            h  = Rb @ g_ref
            h  = h / np.linalg.norm(h)

            def h_of(xv):
                qq = q_norm(xv[:4])
                Rb_ = rotmat_from_q(qq).T
                hv  = Rb_ @ g_ref
                return hv / np.linalg.norm(hv)

            H = np.zeros((3,7)); eps = 1e-6
            for i in range(7):
                dx = np.zeros(7); dx[i]=eps
                hp = h_of(x_pred+dx); hm = h_of(x_pred-dx)
                H[:,i] = (hp - hm)/(2*eps)

            S = H @ P_pred @ H.T + Rm + np.eye(3)*epsS
            K = P_pred @ H.T @ np.linalg.solve(S, np.eye(3))
            y = z - h

            x = x_pred + K @ y
            x[:4] = q_norm(x[:4])
            KH = K @ H
            P = (I7 - KH) @ P_pred @ (I7 - KH).T + K @ Rm @ K.T
        else:
            x = x_pred; P = P_pred

        q = x[:4]; b = x[4:]
        Rb = rotmat_from_q(q).T
        acc_lin = acc_k - (Rb @ g_ref)

        roll, pitch, yaw = euler_from_q(q)
        out['qw'][k], out['qx'][k], out['qy'][k], out['qz'][k] = q
        out['bgx'][k], out['bgy'][k], out['bgz'][k] = b
        out['roll'][k], out['pitch'][k], out['yaw'][k] = roll, pitch, yaw
        out['acc_lin_x'][k], out['acc_lin_y'][k], out['acc_lin_z'][k] = acc_lin

    return out

# ---------- MOVE 단위로 EKF 실행 ----------
def run_ekf_by_move(df_raw: pd.DataFrame) -> pd.DataFrame:
    imu = pick_imu_block(df_raw)
    imu = ensure_units(imu)
    move_col = pick_move_col(df_raw)

    N = len(df_raw)
    out_cols = ['qw','qx','qy','qz','roll','pitch','yaw','bgx','bgy','bgz','acc_lin_x','acc_lin_y','acc_lin_z']
    buff = {c: np.full(N, np.nan, dtype=float) for c in out_cols}

    if move_col is None or not REINIT_PER_MOVE:
        # 파일 전체를 한 덩어리로 처리 (이전 동작)
        q0 = estimate_init_q_from_acc(imu[['ax','ay','az']].values[:max(1,int(INIT_STATIC_SEC*FS))].mean(axis=0))
        b0 = imu[['gx','gy','gz']].values[:max(1,int(INIT_STATIC_SEC*FS))].mean(axis=0)
        ekf = ahrs_ekf_stable(imu[['ax','ay','az']].values,
                              imu[['gx','gy','gz']].values,
                              imu['t'].values,
                              q0=q0, b0=b0)
        for c in out_cols: buff[c][:] = ekf[c]
        return pd.DataFrame(buff)

    # MOVE별 처리
    mv = df_raw[move_col].fillna(method='ffill').fillna(method='bfill').values
    # 원본 인덱스를 기준으로 그룹화(순서 보존)
    idx = np.arange(N)
    for mid, seg_idx in pd.Series(idx).groupby(mv):
        seg_idx = seg_idx.values
        # IMU/시간을 해당 구간으로 추출
        acc  = imu.loc[seg_idx, ['ax','ay','az']].values
        gyro = imu.loc[seg_idx, ['gx','gy','gz']].values
        tt   = imu.loc[seg_idx, 't'].values

        # MOVE 시작부에서 초기값 추정
        L0 = max(1, int(INIT_STATIC_SEC * FS))
        a0 = acc[:L0].mean(axis=0)
        g0 = gyro[:L0].mean(axis=0)
        q0 = estimate_init_q_from_acc(a0)
        b0 = g0

        ekf = ahrs_ekf_stable(acc, gyro, tt, q0=q0, b0=b0)
        for c in out_cols:
            buff[c][seg_idx] = ekf[c]

    return pd.DataFrame(buff)

def append_ekf_to_one_file(in_path: Path, out_dir: Path) -> tuple[bool,str,int]:
    try:
        df_raw = read_csv_safely(in_path)
        append_df = run_ekf_by_move(df_raw)

        # 길이 맞춰 병합
        n = min(len(df_raw), len(append_df))
        df_out = pd.concat([df_raw.iloc[:n].reset_index(drop=True),
                            append_df.iloc[:n].reset_index(drop=True)], axis=1)

        out_rel_dir = in_path.parent.relative_to(DATASET_ROOT)
        (out_dir / out_rel_dir).mkdir(parents=True, exist_ok=True)
        out_path = (out_dir / out_rel_dir / (in_path.stem + "_ekf.csv"))
        df_out.to_csv(out_path, index=False)
        return True, out_path.name, n
    except Exception as e:
        return False, str(e), 0

def process_dataset(dataset_root: Path, out_dir: Path, patterns=('*.csv','*.CSV')):
    logs = []
    files = []
    for pat in patterns:
        files += list(dataset_root.rglob(pat))
    files = sorted(files)
    print(f"[INFO] 대상 CSV: {len(files)}개")

    ok = sk = 0
    for p in files:
        if any(x in p.name for x in ["histogram","label_counts","group_move_counts","meta","standardization_check"]):
            continue
        ok1, msg, n = append_ekf_to_one_file(p, out_dir)
        if ok1:
            ok += 1; print(f"[OK] {p.name} → {msg} (rows={n})")
            logs.append({"file": str(p.relative_to(dataset_root)), "status": "ok", "rows": n, "out": msg})
        else:
            sk += 1; print(f"[SKIP] {p.name} → {msg}")
            logs.append({"file": str(p.relative_to(dataset_root)), "status": "skip", "reason": msg})

    pd.DataFrame(logs).to_csv(out_dir / "ekf_summary.csv", index=False)
    print(f"\n[DONE] 성공 {ok}개 | 스킵 {sk}개 → {out_dir}")

# ---------- 실행 ----------
process_dataset(DATASET_ROOT, OUT_DIR)

# (옵션) 결과 ZIP
ZIP_OUT = EXTRACT_DIR / "ekf_appended_csv_only.zip"
with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in OUT_DIR.rglob("*.csv"):
        zf.write(p, arcname=str(p.relative_to(OUT_DIR)))
print(f"[ZIP] {ZIP_OUT} 생성 완료")


[INFO] 대상 CSV: 2개


/tmp/ipython-input-2714702654.py:247: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  mv = df_raw[move_col].fillna(method='ffill').fillna(method='bfill').values
/tmp/ipython-input-2714702654.py:179: RuntimeWarning: overflow encountered in matmul
  P_pred = F @ P @ F.T + Q
/tmp/ipython-input-2714702654.py:179: RuntimeWarning: invalid value encountered in matmul
  P_pred = F @ P @ F.T + Q
/tmp/ipython-input-2714702654.py:179: RuntimeWarning: overflow encountered in matmul
  P_pred = F @ P @ F.T + Q
/tmp/ipython-input-2714702654.py:179: RuntimeWarning: invalid value encountered in matmul
  P_pred = F @ P @ F.T + Q
/tmp/ipython-input-2714702654.py:179: RuntimeWarning: overflow encountered in matmul
  P_pred = F @ P @ F.T + Q
/tmp/ipython-input-2714702654.py:179: RuntimeWarning: invalid value encountered in matmul
  P_pred = F @ P @ F.T + Q


[OK] IMU_Label_Plus_ALL.csv → IMU_Label_Plus_ALL_ekf.csv (rows=36535)


/tmp/ipython-input-2714702654.py:247: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  mv = df_raw[move_col].fillna(method='ffill').fillna(method='bfill').values


[OK] IMU_label_1_jungro.csv → IMU_label_1_jungro_ekf.csv (rows=8608)

[DONE] 성공 2개 | 스킵 0개 → /content/IMU_DATA_extracted/ekf_appended
[ZIP] /content/IMU_DATA_extracted/ekf_appended_csv_only.zip 생성 완료


In [3]:
# === 옵션 A: 방금 만든 ZIP(결과 CSV만) 즉시 다운로드 ===
from google.colab import files
files.download('/content/IMU_DATA_extracted/ekf_appended_csv_only.zip')  # 한 번에 받기


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>